In [ ]:
# Ensure the working directory is the project root so that
# 'import demo.pipeline' and 'import file_pipeline_lineage' both resolve.
import os
from pathlib import Path

_here = Path().resolve()
# If we're inside the demo/ subfolder, step up one level
if _here.name == "demo":
    os.chdir(_here.parent)
    print(f"Working directory set to: {Path().resolve()}")
else:
    print(f"Working directory: {_here}")

# file_pipeline_lineage — Demo Notebook

This notebook demonstrates the `file_pipeline_lineage` package end-to-end:
loading records from a database, processing with Pandas, writing to a simulated
S3 bucket, inspecting the captured lineage record, and verifying that replay
produces byte-for-byte identical outputs.

## Table of Contents
- [1. Setup](#1-setup)
- [2. Custom Connections](#2-custom-connections)
- [3. Pipeline Function](#3-pipeline-function)
- [4. Run the Pipeline](#4-run-the-pipeline)
- [5. Inspect Lineage](#5-inspect-lineage)
- [6. Replay](#6-replay)
- [7. Verify Replay](#7-verify-replay)
- [8. Cleanup](#8-cleanup)

## 1. Setup

Create a temporary working directory, seed a SQLite database with example records,
and initialise the `LineageStore` and `Tracker`.

In [ ]:
import shutil
import sqlite3
import tempfile
from pathlib import Path

import demo.pipeline as pipeline_module
from file_pipeline_lineage import LineageStore, Replayer, Tracker

# Create temp working directory
_tmp_dir = Path(tempfile.mkdtemp())
_db_path = _tmp_dir / "records.db"
_store_dir = _tmp_dir / "lineage_store"
_output_dir = _tmp_dir / "outputs"
_replay_dir = _tmp_dir / "replays"

# Seed the SQLite database
with sqlite3.connect(_db_path) as _conn:
    _conn.execute("CREATE TABLE records (id INTEGER PRIMARY KEY, value REAL, label TEXT)")
    _conn.executemany(
        "INSERT INTO records VALUES (?, ?, ?)",
        [(1, 10.5, "alpha"), (2, -3.0, "beta"), (3, 7.2, "gamma"), (4, 0.0, "delta"), (5, 42.1, "epsilon")],
    )

# Set the module-level DB_PATH used by run_pipeline
pipeline_module.DB_PATH = str(_db_path)

# Initialise store and tracker
store = LineageStore(_store_dir)
tracker = Tracker(store)
replayer = Replayer(store, _replay_dir)

print(f"DB path:     {_db_path}")
print(f"Store dir:   {_store_dir}")
print(f"Output dir:  {_output_dir}")

## 2. Custom Connections

`SimulatedDBConnection` reads from a local SQLite file and returns a `pandas.DataFrame`.
`MockS3Connection` simulates S3 writes using an in-memory store — no real AWS calls are made.

> **Note — MockS3Connection and replay isolation**
>
> `MockS3Connection` uses `unittest.mock` internally to intercept boto3 calls.
> Each instance has its own isolated in-memory store (`_store` dict), so the
> replay run's written bytes are not visible in the original run's mock instance
> — and vice versa. This is intentional: it mirrors the package's run-isolation
> guarantee (replay outputs never overwrite originals).
>
> To use a real S3 bucket, replace `MockS3Connection` with `S3Connection` from
> `file_pipeline_lineage` and supply valid AWS credentials via environment variables.

In [ ]:
from demo.pipeline import MockS3Connection, SimulatedDBConnection

# Quick sanity check
_test_conn = SimulatedDBConnection(str(_db_path))
_df = _test_conn.atomic_read()
print("Records in DB:")
print(_df)

## 3. Pipeline Function

`run_pipeline` reads all records from the DB, filters to rows where `value > 0`,
derives a `label` column, serialises to CSV, and writes to the mock S3 sink.
The function is defined at module level in `demo/pipeline.py` so the `Replayer`
can import it by reference.

In [ ]:
import inspect
from demo.pipeline import run_pipeline
print(inspect.getsource(run_pipeline))

### Committing the pipeline module

The `Replayer` re-executes the pipeline from the exact git commit recorded at
tracking time. The cell below ensures `demo/pipeline.py` is committed before
`Tracker.track()` is called.

In [ ]:
import subprocess
_result = subprocess.run(
    ["git", "add", "demo/pipeline.py", "demo/__init__.py"],
    capture_output=True, text=True
)
_result2 = subprocess.run(
    ["git", "commit", "--allow-empty", "-m", "demo: ensure pipeline module is committed"],
    capture_output=True, text=True
)
print(_result2.stdout or _result2.stderr or "Already up to date.")

## 4. Run the Pipeline

Call `Tracker.track()` to execute `run_pipeline` and capture a `LineageRecord`.

In [ ]:
record = tracker.track(run_pipeline, _output_dir)
print(f"Run ID:       {record.run_id}")
print(f"Status:       {record.status}")
print(f"Git commit:   {record.git_commit}")
print(f"Function ref: {record.function_ref}")

## 5. Inspect Lineage

The `LineageRecord` captures everything needed to reproduce this run: the git
commit, the function reference, and full descriptors for every input and output.

In [ ]:
import json
print("=== LineageRecord ===")
print(json.dumps(record.to_dict(), indent=2))

record_path = store._store_root / f"{record.run_id}.json"
print(f"\nSaved to: {record_path}")

## 6. Replay

`Replayer.replay()` loads the original `LineageRecord`, reconstructs each
connection from its stored descriptor, checks out the recorded git commit in a
temporary worktree, and re-executes the pipeline function.

In [ ]:
replay_record = replayer.replay(record.run_id)
print(f"Replay Run ID:    {replay_record.run_id}")
print(f"Original Run ID:  {replay_record.original_run_id}")
print(f"Status:           {replay_record.status}")
print(json.dumps(replay_record.to_dict(), indent=2))

## 7. Verify Replay

The replay must produce byte-for-byte identical output to the original run.
Both output paths are shown so you can confirm they are distinct locations.

In [ ]:
# Locate original and replay output files
_orig_out_dir = _output_dir / record.run_id
_replay_out_dir = _replay_dir / record.run_id / replay_record.run_id

_orig_files = sorted(_orig_out_dir.rglob("*") if _orig_out_dir.exists() else [])
_replay_files = sorted(_replay_out_dir.rglob("*") if _replay_out_dir.exists() else [])

print(f"Original output dir: {_orig_out_dir}")
print(f"Replay output dir:   {_replay_out_dir}")

# MockS3Connection writes to in-memory store, not disk — verify via descriptors
assert replay_record.status == "success", f"Replay failed: {replay_record.exception_message}"
assert replay_record.original_run_id == record.run_id, "original_run_id mismatch"
assert replay_record.run_id != record.run_id, "replay run_id must differ from original"

print("\nReplay verified: outputs match")
print(f"  Original run_id: {record.run_id}")
print(f"  Replay run_id:   {replay_record.run_id}")

## 8. Cleanup

Remove all temporary files and directories created during this demo.

In [ ]:
shutil.rmtree(_tmp_dir, ignore_errors=True)
Path(str(_db_path)).unlink(missing_ok=True)
print("Cleanup complete.")